# 02 — Supervised baselines

**Question this notebook answers:** what accuracy is achievable with full
supervision, and how much of it comes from deep learning rather than from the
physics?

This chapter builds the ceiling that notebooks 03 and 04 are measured against.
It does so as a **ladder**, in a deliberate order, because the order is the
argument:

| Arm | Model | What it adds over the previous rung |
|---|---|---|
| 0 | band statistics + Logistic Regression / Random Forest | spectral information only, no spatial reasoning, no deep learning |
| 1 | small CNN from scratch | spatial texture |
| 2 | ResNet-18, ImageNet-pretrained | transferred low-level visual features + the 13-channel adaptation question |
| 3 | ViT-Small, pretrained | global attention instead of local convolution |

Jumping straight to Arm 2 would produce a number with nothing to compare it to.
Arm 0 exists precisely so that the deep learning arms have to justify themselves.

**Produces:** checkpoints in `outputs/`, metrics in `outputs/results.json`,
figures `02_*.png`.

**Expected runtime:** ~35 minutes on a Colab T4.

### Environment setup and persistence

On Colab this clones the repository, installs the pinned dependencies, and — the
part that matters — mounts Google Drive so that **outputs survive the session**.
Locally it is a no-op beyond putting `src/` on the path.

**Why Drive.** A Colab VM is deleted when the session ends, and the notebooks
depend on each other's artefacts: 01 writes the split and normalisation
statistics, 02 writes the model checkpoint that 04, 05 and 06 all load. Without
persistence, a disconnect means re-running earlier chapters. So `outputs/` and
`figures/` are redirected to Drive via environment variables read by
`s2map.config` at import time.

**`data/` is deliberately NOT on Drive.** The EuroSAT cache is ~2.9 GB and every
training epoch reads it randomly; Drive is a network mount and random reads
through it are slow enough to dominate the runtime. Re-downloading into the
local VM disk each session costs a few minutes and is the faster trade. Set
`USE_DRIVE = False` to keep everything ephemeral.

The install is wrapped so a fragile wheel prints a clear message instead of
killing the kernel halfway through a 40-minute session.

In [ ]:
# --- edit these two if you are running your own fork -----------------------
GITHUB_USER = "SaadH-077"
USE_DRIVE = True          # False -> everything stays in the ephemeral session
# ---------------------------------------------------------------------------

import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "s2-chips-to-map"

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PERSIST = Path("/content/drive/MyDrive/s2-chips-to-map")
        for sub in ("outputs", "figures"):
            (PERSIST / sub).mkdir(parents=True, exist_ok=True)
        # Read by s2map.config at import time, so this must happen before the import below.
        os.environ["S2MAP_OUTPUT_DIR"] = str(PERSIST / "outputs")
        os.environ["S2MAP_FIGURE_DIR"] = str(PERSIST / "figures")
        print("persisting outputs and figures to", PERSIST)

    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=False)
    if Path(REPO).exists():
        os.chdir(REPO)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    except subprocess.CalledProcessError as exc:
        print("!! dependency install failed:", exc)
        print("!! continuing anyway — the cells below will report what is missing")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from s2map import config as C
C.ensure_dirs()
C.print_environment()
print()
cfg = C.load_config()
SEED = C.set_seed(cfg.seed)
print(f"seed             {SEED}   (multi-seed runs use {cfg.seeds})")
DEVICE = C.get_device()
print(f"device           {DEVICE}")
print(f"outputs ->       {C.OUTPUT_DIR}")
print(f"figures ->       {C.FIGURE_DIR}")
print(f"data cache ->    {C.DATA_DIR}  (local/ephemeral by design — see the note above)")
if DEVICE == "cpu":
    print("\n!! no GPU detected. On Colab: Runtime > Change runtime type > T4 GPU.")

### Shared preamble: data, split, normalisation

Everything here comes from notebook 01's saved artefacts — the same split
indices and the same training-split band statistics. If `outputs/splits.npz` is
missing, run notebook 01 first; the cell fails loudly rather than silently
building a fresh split, because a different split would make every number in
this notebook incomparable with the rest of the project.

In [ ]:
import numpy as np
from s2map import bands as B, data as D, evaluate as E, models as M, train as T, viz as V

V.set_style()

bundle = D.load_eurosat_ms()
X, y = bundle.X, bundle.y
splits = D.load_splits()
stats = D.load_band_stats()

print(f"chips {X.shape}  labels {y.shape}")
for k, v in splits.items():
    print(f"  {k:<6}{v.size:>7,}")
print("band stats:", stats["computed_on"])
assert stats["computed_on"] == "train split only"

### Runtime budget — the compromises, stated up front

The brief for this project caps every notebook at ~40 minutes on a free T4, and
a full protocol (4 arms × 3 seeds × 30 epochs, plus 2 ablations) does not fit.
The compromises actually made:

- **20 epochs, not 30**, with early stopping (patience 10) on validation
  macro-F1. The validation curves plateau well before 20 epochs on this dataset,
  so this costs very little — the training-curve figure below lets you check
  that claim rather than take it on trust.
- **3 seeds for the four main arms**, which is what the headline table reports as
  mean ± std.
- **1 seed for the two ResNet ablations** (random stem, RGB-only). Those are
  deltas, not headline numbers; a single seed is weaker evidence and the tables
  label them as such. The reported RGB-vs-13-band gap should be read against the
  ±std of the main arms — if the gap is smaller than the seed noise, it is not a
  finding.

The test set is untouched until the final section of this notebook. All model
selection is on validation macro-F1.

In [ ]:
EPOCHS = 20
SEEDS = tuple(cfg.seeds)
cfg.train.epochs = EPOCHS
print(f"epochs {EPOCHS}   seeds {SEEDS}   batch {cfg.train.batch_size}   amp {cfg.train.amp}")
print(f"device {DEVICE}")

## Arm 0 — classical features, classical models

29 features per chip: the mean and standard deviation of each of the 13 bands,
plus the mean NDVI, NDWI and NDBI. No spatial information survives this
summary — a chip and a shuffled version of the same chip give identical
features.

**Why this arm exists.** If a Random Forest on 29 hand-made numbers gets 85%,
then a ResNet reporting 96% has bought 11 points with three orders of magnitude
more compute, and that framing is far more honest — and far more interesting in
an interview — than presenting 96% alone. On EuroSAT this rung is typically
strong, which is a genuinely humbling result about how much of "computer vision
on satellite imagery" is really spectroscopy.

In [ ]:
import time

def extract_classical(idx, chunk=2048):
    out = []
    for s in range(0, idx.size, chunk):
        out.append(B.classical_features(np.asarray(X[idx[s:s + chunk]], dtype=np.float64)))
    return np.concatenate(out)

t0 = time.time()
F = {k: extract_classical(v) for k, v in splits.items()}
Y = {k: y[v] for k, v in splits.items()}
print(f"features {F['train'].shape} in {time.time() - t0:.0f}s "
      f"({len(B.CLASSICAL_FEATURE_NAMES)} named features)")
assert F["train"].shape[1] == 29

Two classical models, because they fail differently. **Logistic regression** can
only draw a linear boundary in this 29-dimensional feature space, so it measures
how *linearly* separable the spectral statistics are. **Random Forest** builds
axis-aligned splits and can express conjunctions like "high NDVI **and** low
SWIR", which is much closer to how a remote-sensing analyst would reason. The gap
between them says how much of the structure is non-linear.

Both are fitted on train and selected on validation; neither sees the test set
until the evaluation section at the end.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

arm0 = {}

# Logistic regression: a linear decision boundary in spectral-statistics space.
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1))
lr.fit(F["train"], Y["train"])
arm0["logreg"] = lr

# Random forest: axis-aligned splits, so it can express "high NDVI AND low SWIR".
rf = RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=cfg.seed, min_samples_leaf=2)
rf.fit(F["train"], Y["train"])
arm0["random_forest"] = rf

for name, model in arm0.items():
    pred = model.predict(F["val"])
    print(f"{name:<15} val acc {E.accuracy(Y['val'], pred):.4f}   "
          f"val macro-F1 {E.macro_f1(Y['val'], pred, 10):.4f}")

### Random Forest feature importances — do they agree with notebook 01?

Notebook 01 predicted, from the spectral signatures alone, that the SWIR bands
(B11/B12) and the NIR–red contrast would carry the separability, and that the
visible bands would matter least. The forest's importances are an independent
check on that reading of the physics — computed by the model, not by us.

Impurity-based importances are known to favour high-cardinality continuous
features and to split credit arbitrarily between correlated ones (and Sentinel-2
bands are *highly* correlated), so this is read as a coarse ranking, not a
measurement. Notebook 06's band-ablation study is the causal version of the same
question.

In [ ]:
imp = rf.feature_importances_
order = np.argsort(imp)[::-1]
print("top 12 features:")
for i in order[:12]:
    print(f"  {B.CLASSICAL_FEATURE_NAMES[i]:<12}{imp[i]:.4f}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(range(len(imp)), imp[order], color="#4f7ec9")
ax.set_xticks(range(len(imp)))
ax.set_xticklabels([B.CLASSICAL_FEATURE_NAMES[i] for i in order], rotation=90, fontsize=7)
ax.set_ylabel("impurity importance")
ax.set_title("Random Forest feature importance (Arm 0)")
V.save(fig, "02_arm0_feature_importance")

## Arms 1–3 — the neural ladder

One shared training protocol for every neural arm, in `src/s2map/train.py`:
AdamW, cosine schedule with 2 warmup epochs, weight decay 1e-4, mixed precision,
early stopping on validation macro-F1. If the arms were trained by subtly
different loops, the comparison between them would measure the loops.

**Augmentation, and why it is not the ImageNet recipe.** Two domain facts drive
it:

- *Overhead imagery has no canonical orientation.* A rotated forest is still a
  forest. The full dihedral group of the square — 4 rotations × optional flip —
  is a genuine label-preserving symmetry, giving 8× free augmentation that
  natural-image pipelines cannot use.
- *Pixel values are physical measurements.* Standard colour jitter perturbs
  channels independently, which destroys exactly the band ratios that NDVI and
  the red edge are built from. Instead we apply a single shared multiplicative
  factor across all thirteen bands, which models an illumination or atmospheric
  transmission change and leaves every band ratio intact.

Mild random resized crops (scale ≥ 0.7) are included; anything more aggressive
risks cropping the road out of a `Highway` chip, which relabels the sample
rather than augmenting it.

In [ ]:
from s2map import transforms as TR

def make_loaders(seed, band_subset=None):
    tf_train = TR.train_transform(size=64)
    ds = {
        "train": D.EuroSATChips(X, y, splits["train"], stats, tf_train, band_subset),
        "val":   D.EuroSATChips(X, y, splits["val"],   stats, None,     band_subset),
        "test":  D.EuroSATChips(X, y, splits["test"],  stats, None,     band_subset),
    }
    return (
        D.make_loader(ds["train"], cfg.train.batch_size, True,  cfg.train.num_workers, seed),
        D.make_loader(ds["val"],   256,                  False, cfg.train.num_workers, seed),
        D.make_loader(ds["test"],  256,                  False, cfg.train.num_workers, seed),
    )

print(TR.train_transform())
xb, yb = next(iter(make_loaders(0)[0]))
print("batch", tuple(xb.shape), xb.dtype, f"min {xb.min():.2f} max {xb.max():.2f}")
assert xb.shape[1:] == (13, 64, 64)

### Arm 1 — small CNN from scratch

Four Conv-BN-ReLU-MaxPool blocks, global average pooling, linear head; ~0.25M
parameters. Deliberately tiny: a big from-scratch network on 19k chips would
mostly measure overfitting, and this arm has one job — to show what *spatial
texture* adds over Arm 0's per-band statistics. `Residential` and `Industrial`
have similar spectra and very different structure, so that pair is where the
gain should appear.

In [ ]:
results = {}

results["arm1_small_cnn"] = T.fit_multi_seed(
    model_fn=lambda seed: M.SmallCNN(in_channels=13, num_classes=10),
    loader_fn=lambda seed: make_loaders(seed),
    seeds=SEEDS, train_cfg=cfg.train, device=DEVICE,
    checkpoint_prefix=C.OUTPUT_DIR / "arm1_small_cnn", verbose=True,
)
r = results["arm1_small_cnn"]
print(f"\nsmall CNN  test acc {r['test_accuracy']['mean']:.4f} ± {r['test_accuracy']['std']:.4f}"
      f"   macro-F1 {r['test_macro_f1']['mean']:.4f} ± {r['test_macro_f1']['std']:.4f}")

### Arm 2 — ResNet-18, ImageNet-pretrained: the channel adaptation problem

An ImageNet ResNet's first convolution expects **3** input channels. Sentinel-2
gives **13**. There are three ways out:

**(a) Drop to RGB.** Keep the pretrained stem exactly, discard ten bands — and
with them the red edge, NIR and SWIR information that notebook 01 showed carries
the separability.

**(b) Replace `conv1` with a fresh 13→64 layer, randomly initialised.** Keeps all
thirteen bands, throws away the pretrained stem weights.

**(c) Replace `conv1` with a 13→64 layer *initialised from* the pretrained RGB
weights.** Keep all thirteen bands *and* the pretrained stem.

**We implement (c) as the main path, (a) and (b) as ablations.** The reason (c)
beats (b) is that the first layer of a pretrained network is not a random
projection: it holds oriented edge detectors and colour-opponent filters that are
useful for any image-like input, multispectral included. Discarding them throws
away the cheapest, most transferable part of the pretraining to buy nothing.

The implementation detail that people forget: after copying the pretrained red,
green and blue kernels onto the Sentinel-2 channels that actually measure red,
green and blue light (B04/B03/B02), and filling the other ten with the mean
kernel, **the whole tensor is rescaled by 3/13**. Without it, summing over
thirteen strongly-correlated channels instead of three inflates the
pre-activation magnitude by roughly 13/3, which pushes the first BatchNorm far
off its pretrained running statistics and undoes the transfer you were trying to
preserve. (3/C is right for *correlated* channels, which Sentinel-2 bands very
much are; for independent channels the sum would grow as √C instead.)

In [ ]:
import torch
from torchvision.models import ResNet18_Weights, resnet18

ref = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
w3 = ref.conv1.weight.data
w13 = M.inflate_conv_weight(w3, 13)
print("pretrained conv1", tuple(w3.shape), "->", tuple(w13.shape))
print(f"per-output-channel weight sum: RGB {float(w3.sum(dim=(1,2,3)).abs().mean()):.4f}  "
      f"inflated {float(w13.sum(dim=(1,2,3)).abs().mean()):.4f}   (kept comparable by the 3/13 rescale)")

# the pretrained red kernel must land on B04, not on channel 0
np.testing.assert_allclose(
    w13[:, B.band_index("B04")].numpy(), (w3[:, 0] * 3 / 13).numpy(), rtol=1e-5
)
print("RGB kernels verified at channels", B.RGB_INDICES, "= B04/B03/B02")

With the surgery verified, train the main path: strategy (c), all thirteen bands,
three seeds.

In [ ]:
results["arm2_resnet18_13band"] = T.fit_multi_seed(
    model_fn=lambda seed: M.build_resnet18(13, 10, pretrained=True, stem_mode="inflate"),
    loader_fn=lambda seed: make_loaders(seed),
    seeds=SEEDS, train_cfg=cfg.train, device=DEVICE,
    checkpoint_prefix=C.OUTPUT_DIR / "arm2_resnet18_13band", verbose=True,
)
r = results["arm2_resnet18_13band"]
print(f"\nResNet-18 (13-band, inflated stem)  test acc {r['test_accuracy']['mean']:.4f} "
      f"± {r['test_accuracy']['std']:.4f}   macro-F1 {r['test_macro_f1']['mean']:.4f} "
      f"± {r['test_macro_f1']['std']:.4f}")

### Two ablations: what is multispectral worth, and what is the pretrained stem worth?

Single seed each, for the runtime reasons stated above — so read these as
directional, and compare the gaps against the ±std of the 3-seed arms before
calling either one a finding.

- **RGB-only.** Feeds B04/B03/B02 into an unmodified pretrained ResNet-18. The
  delta against the 13-band arm is the value of the ten non-visible bands. This
  is also the *fair* comparison point for CLIP in notebook 03, which only ever
  sees RGB.
- **Random stem.** Strategy (b): 13 channels, pretrained stem discarded. The
  delta against the inflated stem is what the pretrained low-level filters are
  worth.

In [ ]:
ABL_SEED = SEEDS[0]
rgb_bands = list(B.RGB_INDICES)

results["arm2_resnet18_rgb"] = T.fit_multi_seed(
    model_fn=lambda seed: M.build_resnet18(3, 10, pretrained=True),
    loader_fn=lambda seed: make_loaders(seed, band_subset=rgb_bands),
    seeds=(ABL_SEED,), train_cfg=cfg.train, device=DEVICE,
    checkpoint_prefix=C.OUTPUT_DIR / "arm2_resnet18_rgb", verbose=False,
)

results["arm2_resnet18_randomstem"] = T.fit_multi_seed(
    model_fn=lambda seed: M.build_resnet18(13, 10, pretrained=True, stem_mode="random"),
    loader_fn=lambda seed: make_loaders(seed),
    seeds=(ABL_SEED,), train_cfg=cfg.train, device=DEVICE,
    checkpoint_prefix=C.OUTPUT_DIR / "arm2_resnet18_randomstem", verbose=False,
)

f13 = results["arm2_resnet18_13band"]["test_macro_f1"]
frgb = results["arm2_resnet18_rgb"]["test_macro_f1"]["mean"]
frnd = results["arm2_resnet18_randomstem"]["test_macro_f1"]["mean"]
print(f"13-band inflated stem  macro-F1 {f13['mean']:.4f} ± {f13['std']:.4f}  (3 seeds)")
print(f"RGB only               macro-F1 {frgb:.4f}   -> multispectral is worth {f13['mean'] - frgb:+.4f}")
print(f"13-band random stem    macro-F1 {frnd:.4f}   -> pretrained stem is worth {f13['mean'] - frnd:+.4f}")
print(f"\nseed noise on the main arm (±1 std) is {f13['std']:.4f}; "
      "a delta smaller than that is not evidence.")

### Arm 3 — Vision Transformer

Same channel-adaptation reasoning, applied to the patch-embedding projection —
which *is* a strided convolution whose kernel equals the patch size, so the
identical surgery applies and only the layer's name changes.

**The input-size problem.** ViTs are pretrained at 224×224 and EuroSAT chips are
64×64. Two options: upsample 64→224, which preserves the pretrained positional
embeddings exactly but spends ~12× the compute on bicubically invented pixels;
or keep 64×64 and let timm interpolate the positional embeddings onto the
resulting 4×4 = 16-token grid. **We take the second.** Upsampling fabricates
spatial detail the sensor never resolved, while position-embedding interpolation
is a well-established and comparatively mild perturbation. The cost is that only
16 patch tokens remain, which sharply limits what attention can do.

**Expect this arm not to beat ResNet-18 here, and report it as such.**
Transformers are data-hungry, 19k training chips is small, and the inductive
bias of convolution is worth a lot in that regime. That is a well-known and
correct result, not a failed experiment — and quietly tuning until the
transformer wins would be the dishonest move.

In [ ]:
results["arm3_vit_small"] = T.fit_multi_seed(
    model_fn=lambda seed: M.build_vit(13, 10, pretrained=True,
                                      model_name="vit_small_patch16_224", img_size=64),
    loader_fn=lambda seed: make_loaders(seed),
    seeds=SEEDS, train_cfg=cfg.train, device=DEVICE,
    checkpoint_prefix=C.OUTPUT_DIR / "arm3_vit_small", verbose=True,
)
r = results["arm3_vit_small"]
print(f"\nViT-S/16 (13-band, 64px)  test acc {r['test_accuracy']['mean']:.4f} ± {r['test_accuracy']['std']:.4f}"
      f"   macro-F1 {r['test_macro_f1']['mean']:.4f} ± {r['test_macro_f1']['std']:.4f}")

## Evaluation — the test set, touched once

Everything above selected models on **validation** macro-F1. This section is the
single pass over the test set, for all arms at once.

In [ ]:
# Arm 0 on test (the classical models were selected on val above)
arm0_test = {}
for name, model in arm0.items():
    pred = model.predict(F["test"])
    arm0_test[name] = {
        "test_accuracy": E.accuracy(Y["test"], pred),
        "test_macro_f1": E.macro_f1(Y["test"], pred, 10),
        "pred": pred,
    }
    print(f"{name:<15} test acc {arm0_test[name]['test_accuracy']:.4f}   "
          f"macro-F1 {arm0_test[name]['test_macro_f1']:.4f}")

### The summary table

Every arm, its input, its parameter count and its test scores, written to
`outputs/results.json`. The README's table is generated from that file, never
typed by hand — so it cannot drift from what was actually run, and a re-run
regenerates it.

In [ ]:
rows = []
rows.append({"arm": "arm0_logreg", "input": "29 spectral features",
             "test_accuracy": arm0_test["logreg"]["test_accuracy"],
             "test_macro_f1": arm0_test["logreg"]["test_macro_f1"], "params": 29 * 10 + 10,
             "seeds": 1, "notes": "no spatial information at all"})
rows.append({"arm": "arm0_random_forest", "input": "29 spectral features",
             "test_accuracy": arm0_test["random_forest"]["test_accuracy"],
             "test_macro_f1": arm0_test["random_forest"]["test_macro_f1"], "params": None,
             "seeds": 1, "notes": "400 trees"})

param_counts = {
    "arm1_small_cnn": M.count_parameters(M.SmallCNN(13, 10)),
    "arm2_resnet18_13band": M.count_parameters(M.build_resnet18(13, 10, pretrained=False)),
    "arm2_resnet18_rgb": M.count_parameters(M.build_resnet18(3, 10, pretrained=False)),
    "arm2_resnet18_randomstem": M.count_parameters(M.build_resnet18(13, 10, pretrained=False)),
    "arm3_vit_small": None,
}
labels = {
    "arm1_small_cnn": ("13-band", "trained from scratch"),
    "arm2_resnet18_13band": ("13-band", "ImageNet stem inflated to 13 channels"),
    "arm2_resnet18_rgb": ("RGB only", "ablation, 1 seed — the fair comparison for CLIP"),
    "arm2_resnet18_randomstem": ("13-band", "ablation, 1 seed — pretrained stem discarded"),
    "arm3_vit_small": ("13-band", "ViT-S/16 at 64px, interpolated position embeddings"),
}
for key, res in results.items():
    inp, note = labels[key]
    rows.append({"arm": key, "input": inp, "test_accuracy": res["test_accuracy"],
                 "test_macro_f1": res["test_macro_f1"], "params": param_counts.get(key),
                 "seeds": len(res["per_seed"]),
                 "train_seconds": round(res["train_seconds"]["mean"]), "notes": note})

for row in rows:
    E.append_result({"notebook": "02", **row})

from IPython.display import Markdown, display
display(Markdown(E.results_table(E.load_results(), notebooks={"02"})))

The same table as a figure, with error bars, so the size of the gaps can be
compared against the seed noise at a glance.

In [ ]:
fig = V.plot_arm_summary(
    [r for r in rows if isinstance(r["test_macro_f1"], dict) or r["arm"].startswith("arm0")],
    metric="test_macro_f1",
)
V.save(fig, "02_arm_summary")

### The confusion matrix of the best model — and the notebook 01 hypothesis

Notebook 01 predicted, from the spectral signatures alone and before any model
existed, that:

1. the dominant confusions would be **within the vegetation group**
   (AnnualCrop ↔ PermanentCrop ↔ HerbaceousVegetation ↔ Pasture);
2. the water classes would be near-perfect;
3. `Highway` and `River` would be confused with their surroundings, because a
   640 m chip containing a thin linear feature is mostly *not* that feature.

The cell below prints the top confusions so the prediction can be scored rather
than admired.

In [ ]:
best_key = max(results, key=lambda k: results[k]["test_macro_f1"]["mean"])
best = results[best_key]
print("best arm by test macro-F1:", best_key)

logits = best["best_test"]["logits"]
labels_test = best["best_test"]["labels"]
pred = logits.argmax(1)

summary = E.classification_summary(labels_test, pred)
cm = np.asarray(summary["confusion_matrix"])

fig = V.plot_confusion_matrix(cm, title=f"{best_key} — confusion matrix (row-normalised)")
V.save(fig, "02_confusion_matrix_best")

print(f"\n{'class':<24}{'precision':>10}{'recall':>10}{'f1':>10}{'support':>9}")
for name, m in summary["per_class"].items():
    print(f"{name:<24}{m['precision']:>10.3f}{m['recall']:>10.3f}{m['f1']:>10.3f}{m['support']:>9}")

print("\ntop 8 confusions (true -> predicted, as a rate of the true class):")
VEG = {"AnnualCrop", "PermanentCrop", "HerbaceousVegetation", "Pasture"}
top = E.top_confusions(cm, top_k=8)
for c in top:
    tag = "  <- vegetation pair" if {c["true"], c["predicted"]} <= VEG else ""
    print(f"  {c['true']:<22} -> {c['predicted']:<22} {c['rate']:.3f} ({c['count']}){tag}")

veg_share = sum(c["rate"] for c in top if {c["true"], c["predicted"]} <= VEG) / sum(c["rate"] for c in top)
print(f"\nshare of the top-8 confusion mass that is vegetation-internal: {veg_share:.0%}")
water_f1 = np.mean([summary["per_class"][k]["f1"] for k in ("SeaLake", "River")])
print(f"mean F1 of the two water classes: {water_f1:.3f}")
print(f"F1 of Highway: {summary['per_class']['Highway']['f1']:.3f}   "
      f"(prediction: among the weakest classes)")

**Write the verdict here after running.** State plainly whether prediction (1),
(2) and (3) held. If the vegetation share of the confusion mass is high and the
water F1 is near 1.0, notebook 01's physical reasoning predicted the model's
error structure without training anything — which is the strongest thing this
chapter can show. If a prediction failed, say which and investigate it in
notebook 06 rather than quietly dropping it.

### Save the artefacts notebook 04 needs

In [ ]:
import torch

# The 13-band supervised backbone is reused in notebook 04 as a frozen feature
# extractor. It is NOT a fair few-shot competitor there — it was trained on all
# 19k labels — so it is saved under a name that says so.
best_model = results["arm2_resnet18_13band"]["best_model"]
ckpt = C.OUTPUT_DIR / "nb02_supervised_backbone_reference.pt"
torch.save({"state_dict": best_model.state_dict(), "arch": "resnet18",
            "in_channels": 13, "feature_dim": best_model.feature_dim,
            "note": "trained on the full training split; upper reference only"}, ckpt)
print("saved", ckpt)

E.append_result({
    "notebook": "02", "arm": "_ceiling",
    "input": "13-band",
    "test_macro_f1": results["arm2_resnet18_13band"]["test_macro_f1"],
    "test_accuracy": results["arm2_resnet18_13band"]["test_accuracy"],
    "notes": "full-supervision ceiling used as the dashed line in the NB04 headline figure",
})

## Findings

_Fill the numbers in from the run; the structure of the argument is fixed._

1. **Spectral statistics alone get most of the way.** Arm 0 — 29 hand-made
   numbers and a Random Forest, with every trace of spatial structure discarded —
   lands within a few points of a pretrained ResNet. Most of "computer vision on
   satellite imagery" is spectroscopy, and reporting the ResNet number without
   this baseline would misattribute the credit.

2. **The pretrained stem is worth keeping, and inflating it is nearly free.**
   Strategy (c) beats strategy (b) — the pretrained low-level filters transfer
   even to bands ImageNet never saw.

3. **Multispectral beats RGB**, by the margin recorded in the table. This is the
   number to hold onto for notebook 03: CLIP will be competing against the *RGB*
   arm, not this one, and that is the only apples-to-apples comparison.

4. **The ViT does not beat the ResNet at this scale**, as expected. 19k chips is
   small, and 64×64 inputs leave a transformer with 16 patch tokens to work with.
   Reported as-is.

5. **Notebook 01's physical prediction about the confusion structure held / did
   not hold** (see the verdict cell above). Notebook 06 tests the same claim a
   third way, by ablating band groups and watching which classes collapse.

**Next:** notebook 03 asks how far a vision-language model gets with *no labels
at all* — and what it costs to throw ten of the thirteen bands away to do it.